# Affect and Attention-Aware Emotion Detection
### Model Training and Performance Report
This notebook trains a Lightweight Random Forest Classifier using purely numerical facial features (Eye Openness, Eyebrow Distance, Mouth Opening, Head Tilt) extracted via MediaPipe. 

This methodology strictly aligns with **Section 3.2.3** of the Research Proposal, ensuring privacy-preserving, computationally efficient, real-time emotion detection without processing raw images.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Load the dataset
df = pd.read_csv('extracted_features.csv')
print(f"Loaded dataset with {len(df)} samples.")
df.head()

In [ ]:
# Prepare the exact 4 features requested by the proposal
X = df[['eye_openness', 'eyebrow_distance', 'mouth_opening', 'head_tilt']]
y = df['emotion']

# Encode text labels to numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("Training Lightweight Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {acc*100:.2f}%")

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Emotion Classification Confusion Matrix')
plt.ylabel('True Emotion')
plt.xlabel('Predicted Emotion')
plt.show()

In [ ]:
# Feature Importance Analysis
importances = rf_model.feature_importances_
features = X.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
sns.barplot(x=[features[i] for i in indices], y=importances[indices], palette='viridis')
plt.title('Feature Importances for Emotion Detection')
plt.ylabel('Relative Importance')
plt.xlabel('Facial Geometry Features')
plt.show()